# CV behaviour visualisation

In [ ]:
def multilabel_binarise (label):
    classes = np.unique(label)
    new_label = label
    for i in range (0,len(classes)):
        class_i = classes[i]
        new_label[new_label==class_i] = i
    return new_label


def plot_cv(cv, X, old_label, old_group, n_splits, lw=10):
    cmap_data = plt.cm.Paired
    cmap_cv = plt.cm.coolwarm
    fig, ax = plt.subplots()
    for ii, (tr, tt) in enumerate(cv.split(X,old_label)):
        # Fill in indices with the training/test groups
        indices = np.array([np.nan] * len(X))
        indices[tt] = 1
        indices[tr] = 0

        # Visualize the results
        ax.scatter(
            range(len(indices)),
            [ii + 0.5] * len(indices),
            c=indices,
            marker="_",
            lw=lw,
            cmap=cmap_cv,
            vmin=-0.2,
            vmax=1.2,
        )
    new_label = multilabel_binarise (old_label)
    new_group = multilabel_binarise (old_group)
    ax.scatter(
            range(len(X)), [ii + 1.5] * len(X), c=new_label, marker="_", lw=lw, cmap=cmap_data
        )
    ax.scatter(
        range(len(X)), [ii + 2.5] * len(X), c=new_group, marker="_", lw=lw, cmap=cmap_data
    )
    # Formatting
    yticklabels = list(range(n_splits)) + ["class", "group"]
    ax.set(
        yticks=np.arange(n_splits + 2) + 0.5,
        yticklabels=yticklabels,
        xlabel="Sample index",
        ylabel="CV iteration",
    )
    ax.set_title("{}".format(type(cv).__name__), fontsize=15)
    return ax

# function to calculate confusion matrix

In [ ]:
def calculate_cm (cv_results, label):
    results=pd.DataFrame(index=range(0,len(label)), columns=['LOOCV results','labels'])
    results.iloc[:,0]=cv_results
    results.iloc[:,1]=label
    labelnames = set(label)
    labelnames = list(labelnames)
    dims = len(labelnames)
    
    #calculate diagonals for TPs & TNs
    TRUE = []
#     TN = []
    for i in range (0,dims):
        condition1=(results['LOOCV results']==1) & (results['labels'] == labelnames[i])
        dum=results[condition1]
        TRUE.append(len(dum))
    
    FALSE = []
    #calculate other elements
    for i in range (0,dims):
        condition2=(results['LOOCV results']==0) & (results['labels'] == labelnames[i])
        dum=results[condition2]
        FALSE.append(len(dum))
    c_matrix=np.zeros((dims,dims))
    for i in range (0, dims):
        for j in range (0, dims):
            if i == j:
                c_matrix[i,j]= TRUE [i]
            else:
                c_matrix[i,j]= FALSE [i]
    return c_matrix

# function to calculate F1/pre/acc/sen/spe

In [ ]:
def calculate_FPASS (cm,cm_label):
    dims = len(cm_label)
    results=pd.DataFrame(index=range(0,dims+1), columns=['class','accuracy','sensitivity','specificity','precision','F1'])
    results.iloc[:-1,0]=cm_label

    
    for i in range(dims):
        TP = cm[i][i]
        FP = np.sum(cm[i,:])-TP
        FN = np.sum(cm[:,i])-TP
        TN = np.sum(np.sum(cm))-TP-FP-FN
        results.iloc[i,2] = (TP)/(TP+FN)
        results.iloc[i,3] = (TN)/(TN+FP)
        results.iloc[i,4] = (TP)/(TP+FP)
        results.iloc[i,5] = 2*(TP)/(2*TP+FP+FN)
        results.iloc[i,1] = (results.iloc[i,2]+results.iloc[i,3])/2
    
    results.iloc[i+1,0]='Average'
    results.iloc[i+1,1] = np.mean(results.iloc[:,1])
    results.iloc[i+1,2] = np.mean(results.iloc[:,2])
    results.iloc[i+1,3] = np.mean(results.iloc[:,3])
    results.iloc[i+1,4] = np.mean(results.iloc[:,4])
    results.iloc[i+1,5] = np.mean(results.iloc[:,5])
    return results

# function to box plot

In [ ]:
def box_plot (features,mz,box_data,data_label, save = False, savepath = None):
    group_label = list(box_data.columns)
#     for i in range(group_label):
    #         colors = {"APC_control": "g", "APC_sim": "g","APC_KRAS_control": "r", "APC_KRAS_sim": "r",
#                  "APC_P53_control": "y", "APC_P53_sim": "y","APC_KRAS_P53_control": "b", "APC_KRAS_P53_sim": "b"}
    for feature in features:
        index= np.argmin(abs(mz-float(feature)))
        fig = plt.figure(figsize=(15,8))
        sns.boxplot(x=data_label, y=box_data.columns[index+2], data=box_data, whis=5, palette = 'Set1')
        if save == True:
            filename = 'BP_'+box_data.columns[index+2]
            path = os.path.join(savepath, 'box_plots')
            if not os.path.exists(path):
                os.mkdir(path)
            os.chdir(path)
            fig.savefig(filename+'.png')
            plt.close()
            print('plot for '+str(feature)+' saved.')
    os.chdir('..')